# 🌻 Perjalanan Yooji dan Mina Menonton Piala Dunia - Gammafest Solution

**Tim**: [ID Tim_Nama Tim]

## Overview

Notebook ini berisi solusi untuk kompetisi Gammafest dengan tujuan memprediksi skor pertandingan sepak bola internasional (`team_goals` dan `opp_goals`) menggunakan dataset historis dari tahun 1872 hingga 2026.

## Metrik Evaluasi: AW-MAE (Augmented Weighted Mean Absolute Error)

Metrik ini mempertimbangkan:
- Ketepatan skor akhir (Exact Score Penalty: 0.30)
- Ketepatan hasil pertandingan (Outcome Penalty: 0.25)
- Ketepatan selisih gol (Goal Difference Penalty: 0.15)
- Outcome Multiplier (1.5x jika outcome salah)
- Non-Linear Scaling (pangkat 1.5)
- Tournament Weighting (bobot berbeda per turnamen)

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Load Data
print("Loading data...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")

In [ ]:
# Explore Training Data
print("\n=== Training Data Info ===")
print(train.info())
print("\n=== First 5 rows ===")
display(train.head())
print("\n=== Target Distribution ===")
print(train['team_goals'].describe())
print(train['opp_goals'].describe())

In [ ]:
# Explore Test Data
print("\n=== Test Data Info ===")
print(test.info())
print("\n=== First 5 rows ===")
display(test.head())

# Check columns available in test but not in train (performance features missing)
train_cols = set(train.columns)
test_cols = set(test.columns)
print(f"\nColumns in train but not in test: {train_cols - test_cols}")

In [ ]:
# Feature Engineering
# Features available in both train and test
common_features = ['is_home', 'neutral', 'confederation_team', 'confederation_opp', 
                   'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
                   'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']

def prepare_features(df, is_train=True):
    df = df.copy()
    
    # Fill missing values with median
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    # Encode confederation categorical variables
    le_conf_team = LabelEncoder()
    le_conf_opp = LabelEncoder()
    
    all_conf_team = list(df['confederation_team'].fillna('Unknown').unique())
    all_conf_opp = list(df['confederation_opp'].fillna('Unknown').unique())
    
    le_conf_team.fit(all_conf_team)
    le_conf_opp.fit(all_conf_opp)
    
    df['confederation_team_enc'] = le_conf_team.transform(df['confederation_team'].fillna('Unknown'))
    df['confederation_opp_enc'] = le_conf_opp.transform(df['confederation_opp'].fillna('Unknown'))
    
    # Gender encoding (M=1, F=0)
    df['gender_enc'] = (df['gender'] == 'M').astype(int)
    
    return df

train_processed = prepare_features(train, is_train=True)
test_processed = prepare_features(test, is_train=False)

print("Feature engineering completed!")

In [ ]:
# Select features for modeling
feature_cols = ['is_home', 'neutral', 'confederation_team_enc', 'confederation_opp_enc',
                'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
                'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue',
                'gender_enc']

X_train = train_processed[feature_cols]
y_train_goals = train_processed['team_goals']
y_train_opp = train_processed['opp_goals']

X_test = test_processed[feature_cols]

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Train Models
print("Training models...")

# Using Gradient Boosting Regressor
model_goals = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model_opp = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)

model_goals.fit(X_train, y_train_goals)
print("Model for team_goals trained!")

model_opp.fit(X_train, y_train_opp)
print("Model for opp_goals trained!")

In [ ]:
# Make Predictions
pred_goals = model_goals.predict(X_test)
pred_opp = model_opp.predict(X_test)

# Round predictions to integers
pred_goals = np.round(pred_goals).astype(int)
pred_opp = np.round(pred_opp).astype(int)

# Ensure non-negative predictions
pred_goals = np.maximum(pred_goals, 0)
pred_opp = np.maximum(pred_opp, 0)

print(f"Predictions generated! Shape: {len(pred_goals)}")
print(f"Team goals range: {pred_goals.min()} - {pred_goals.max()}")
print(f"Opp goals range: {pred_opp.min()} - {pred_opp.max()}")

In [ ]:
# Create Submission File
submission = pd.DataFrame({
    'Id': test_processed['Id'],
    'team_goals': pred_goals,
    'opp_goals': pred_opp
})

# Save submission
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved! Shape: {submission.shape}")
print("\nFirst 10 rows:")
display(submission.head(10))

In [ ]:
# Feature Importance Analysis
print("\n=== Feature Importance ===")
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance (goals)': model_goals.feature_importances_,
    'Importance (opp)': model_opp.feature_importances_
}).sort_values('Importance (goals)', ascending=False)

display(importance_df)

## Kesimpulan

Solusi ini menggunakan Gradient Boosting Regressor dengan fitur-fitur yang tersedia di kedua dataset (train dan test). Model dilatih secara terpisah untuk memprediksi `team_goals` dan `opp_goals`.

### Fitur yang Digunakan:
- `is_home`, `neutral`: Informasi lokasi pertandingan
- `confederation_team_enc`, `confederation_opp_enc`: Encoding konfederasi tim
- `population_team`, `population_opp`: Populasi negara
- `gdp_per_capita_team`, `gdp_per_capita_opp`: GDP per kapita
- `altitude_venue`, `temperature_venue`: Kondisi venue
- `distance_travel_team`, `distance_travel_opp`: Jarak perjalanan
- `gender_enc`: Gender pertandingan

### Potensi Improvement:
1. Feature engineering lebih lanjut
2. Ensemble multiple models
3. Hyperparameter tuning dengan cross-validation
4. Menangani imbalance dalam distribusi gol
5. Mempertimbangkan metrik AW-MAE secara eksplisit dalam training